# Chat With Multiple Documents (PDF, DOCX, TXT, PPTX) using AstraDB and LangChain

This notebook builds a Retrieval Augmented Generation system, usually called RAG, that can read multiple documents of different formats and answer questions about their content.

## What is Retrieval Augmented Generation

A normal language model answers a question using only what it memorized while it was trained. It has no way to know about your personal files, your company documents, or anything written after its training ended. Retrieval Augmented Generation solves this problem by giving the language model access to your own documents at the moment you ask a question.

The idea works in two stages.

1. Retrieval stage. When you ask a question, the system searches through your documents and finds the small pieces of text that are most likely to contain the answer.
2. Generation stage. Those retrieved pieces of text are handed to the language model together with your question, and the model writes an answer using that context.

Because the model is grounded in real text taken from your documents, its answers are far more accurate and far less likely to be invented from nothing.

## What this notebook actually does, from start to finish

The notebook follows one continuous, linear path.

1. Install every Python package and system tool that is needed.
2. Import all the libraries used later in the notebook.
3. Load secret credentials for Astra DB from a local environment file.
4. Create a shared embedding model that turns text into numeric vectors.
5. Run a small demonstration that loads PDF files, splits them into chunks, embeds them, stores them in a temporary in memory vector store, and retrieves relevant chunks for a sample question.
6. Build a permanent pipeline that loads PDF files, splits them, embeds them, and stores them in Astra DB, a cloud vector database. A retriever and a full question answering chain are then built on top of this data using a Groq hosted large language model.
7. Extend the pipeline to a whole folder of mixed document types, meaning PDF, DOCX, TXT, and PPTX files together, again storing everything in Astra DB and building a retriever and a question answering chain for it.
8. Ask several example questions that pull answers from different document types, proving that the system can search across all of them at once.

## Running this notebook locally

This version of the notebook is written to run locally inside VS Code or any local Jupyter environment instead of Google Colab. Every Google Colab specific piece, such as reading secrets from google.colab.userdata or using the folder /content/ as the working directory, has been replaced with a local equivalent, such as a .env file and normal local folders on your own computer.


## Step 1: Install the Required Packages

Before writing any real code we need to install every Python package this notebook depends on.

Because this notebook runs locally instead of on Google Colab, two extra tools also need to be installed directly on your machine, not just as Python packages.

1. Poppler, which is required so the Unstructured loader can read PDF files.
2. Tesseract OCR, which is required only if a PDF contains scanned images of text rather than selectable text.

On Windows the easiest way to install both tools is with Conda, since Conda virtual environments are already used for this kind of project.

```
conda install -c conda-forge poppler
conda install -c conda-forge tesseract
```

On macOS you can instead use Homebrew with `brew install poppler tesseract`, and on Ubuntu or Debian Linux you can use `sudo apt-get install poppler-utils tesseract-ocr`.

Run the cell below once to install all the Python packages needed for this notebook. You only need to do this the first time you set up the project.


In [ ]:
# Install every Python package required for this notebook in a single command.
# Keeping this as one line makes it easy to copy straight into a local terminal
# on Windows, macOS, or Linux.

# langchain and langchain community give us the core RAG building blocks.
# langchain openai and langchain groq give us access to hosted language models.
# langchain astradb lets us talk to the Astra DB vector database.
# langchain text splitters gives us tools to break long text into chunks.
# openai and datasets are optional helper libraries kept for flexibility.
# pypdf, unstructured, unstructured pytesseract, pdf2image, and pdfminer.six
# are all used to read text out of PDF, DOCX, TXT, and PPTX files.
# python dotenv lets us load secret keys from a local .env file.
# tiktoken helps with counting tokens, and Cython speeds up some dependencies.

%pip install langchain langchain-community langchain-openai langchain-astradb langchain-text-splitters openai datasets pypdf unstructured "unstructured[pdf]" "unstructured[pptx]" unstructured-pytesseract pdf2image pdfminer.six python-dotenv tiktoken Cython


### A note about Poppler and Tesseract

The Python package unstructured pytesseract only gives Python a way to talk to Tesseract, it does not install the Tesseract program itself. The same is true for pdf2image, which needs the separate Poppler program to actually turn PDF pages into images.

Make sure Poppler and Tesseract were installed using Conda, Homebrew, or apt as shown above before you continue, otherwise the PDF loading steps later in this notebook may fail with an error about a missing program.


## Step 2: Import the Libraries

Now that everything is installed we import every piece that will be used throughout the notebook.

Grouping all imports together in one place, instead of spreading them across many cells, makes it easy to see everything this project depends on at a single glance.


In [ ]:
# Standard library import used for working with file paths and folders.
import os

# python dotenv lets us load secrets, such as API keys and tokens, from a
# local .env file instead of typing them directly into the notebook. This
# keeps credentials out of any code that gets pushed to GitHub.
from dotenv import load_dotenv

# Hugging Face "datasets" library. It is kept here in case you ever want to
# load a ready made dataset instead of, or in addition to, your own local
# documents.
from datasets import load_dataset

# Document loaders. Each of these reads files from disk and converts them
# into LangChain Document objects, which are simply text plus some metadata
# such as the file name and page number.
from langchain_community.document_loaders import (
    PyPDFLoader,             # reads a single PDF file, page by page
    UnstructuredPDFLoader,   # reads a single PDF using the Unstructured library
    DirectoryLoader,         # automatically reads every supported file inside a folder
)

# The base Document class used everywhere in LangChain.
from langchain_core.documents import Document

# Splits long text into smaller overlapping chunks, which keeps each chunk
# small enough to embed and search effectively while still preserving
# context across chunk boundaries.
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All libraries imported successfully.")


## Step 3: Load Astra DB Configuration from a Local Environment File

The original notebook was written for Google Colab and loaded secrets using google.colab.userdata.get. That approach only works inside Google Colab.

For a local VS Code setup, the recommended approach is to store sensitive credentials in a file named `.env` located in the same folder as this notebook. This keeps API credentials separate from your code and prevents them from being accidentally committed to GitHub.

Because this notebook uses the free Hugging Face embedding model `sentence-transformers/all-MiniLM-L6-v2`, an OpenAI API key is not required for embeddings.

Before running the next cell, create a file named `.env` in the same folder as this notebook and add your Astra DB credentials in the following format.

```text
ASTRA_DB_API_ENDPOINT=your_astra_db_api_endpoint_here
ASTRA_DB_APPLICATION_TOKEN=your_astra_db_application_token_here
ASTRA_DB_KEYSPACE=default_keyspace
```

Also add `.env` to your `.gitignore` file so your credentials are never pushed to a public repository.


In [ ]:
import os
from dotenv import load_dotenv

# Load every variable defined inside the local .env file into the current
# environment, so they can be read with os.getenv.
load_dotenv()

# Read the two Astra DB credentials we need.
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

# Check that both credentials were actually found. If either one is
# missing we stop immediately with a clear error message rather than
# letting a confusing failure happen later in the notebook.
missing = [
    name for name, value in [
        ("ASTRA_DB_API_ENDPOINT", ASTRA_DB_API_ENDPOINT),
        ("ASTRA_DB_APPLICATION_TOKEN", ASTRA_DB_APPLICATION_TOKEN),
    ]
    if not value
]

if missing:
    raise ValueError(
        f"Missing required environment variables: {missing}. "
        "Please add them to your local .env file before continuing."
    )

print("Astra DB configuration loaded successfully.")


## Step 4: Create the Embedding Model

An embedding model converts a piece of text into a list of numbers, called a vector, that captures the meaning of that text. Two pieces of text that mean similar things end up with vectors that are close together in that numeric space, while unrelated pieces of text end up far apart.

This single embedding model will be reused everywhere in the notebook, both when document chunks are stored in a vector database and later when a question is turned into a vector so it can be compared against those stored chunks.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Create one shared, free Hugging Face embeddings object. This model runs
# entirely on your own machine and does not require any API key or paid
# subscription.
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Free embedding model ready.")


## Step 5: Quick Demonstration, Loading Multiple PDFs with the Unstructured Loader

This section is a short, self contained demonstration that shows the complete basic retrieval workflow using a temporary, in memory vector store. It is meant purely for experimentation before we move on to the permanent Astra DB pipeline later in the notebook.

The PDF files used in this demonstration live inside a local folder on disk. Update the folder path in the next cell so it points to wherever your own PDF files are stored on your computer.

Each PDF file is loaded with `UnstructuredPDFLoader`, the extracted text is split into smaller chunks with `RecursiveCharacterTextSplitter`, those chunks are converted into vectors with the shared Hugging Face embedding model created in Step 4, and finally the chunks and vectors are stored temporarily inside an in memory Chroma vector store.

Being in memory means the data only exists for as long as this notebook session is running. As soon as the notebook is restarted, everything stored here disappears. That is perfectly fine for a quick demonstration, but it is not suitable for a real project, which is why Step 6 introduces Astra DB as a permanent alternative.


### Step 5.1: Set the PDF folder path

In [ ]:
import os

# Local folder containing the PDF files used for this quick demonstration.
# Change this path so it points to the folder on your own computer that
# contains your PDF files.
pdf_folder_path = r"f:\Multimodal-RAG-Systems\Multi_Document_Rag\data"

# List every file inside that folder so we can visually confirm the path
# is correct before continuing.
print("Files found:", os.listdir(pdf_folder_path))


### Step 5.2: Create a loader for each PDF file

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader

# Build one UnstructuredPDFLoader object for every PDF file inside the
# folder. Each loader knows how to read exactly one file, but does not
# actually read anything until its load method is called.
loaders = [
    UnstructuredPDFLoader(
        os.path.join(pdf_folder_path, file_name)
    )
    for file_name in os.listdir(pdf_folder_path)
    if file_name.lower().endswith(".pdf")
]

print(f"Created {len(loaders)} PDF loader(s).")


### Step 5.3: Load all PDF documents

In [ ]:
# Call load on every loader we just created and collect all of the
# resulting Document objects into a single flat list.
documents = []

for loader in loaders:
    documents.extend(loader.load())

print(f"Loaded {len(documents)} document(s).")


### Step 5.4: Split the documents into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split every extracted document into smaller overlapping chunks.
# chunk_size controls the maximum number of characters in each chunk.
# chunk_overlap controls how many characters are shared between two
# consecutive chunks, which helps preserve context across the chunk
# boundary so an idea that spans a boundary is not completely lost.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} text chunks.")


### Step 5.5: Create the in memory vector store

In [ ]:
from langchain_chroma import Chroma

# Convert every chunk into a vector using the shared Hugging Face embedding
# model created in Step 4, then store both the chunks and their vectors
# inside a temporary, in memory Chroma vector store.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

print("In memory vector store created from the PDF files.")


### Step 5.6: Create the retriever

In [ ]:
# Turn the in memory vector store into a retriever. A retriever is a
# simple interface that, given a text query, returns the most relevant
# stored chunks. Here we ask it to always return the top 3 matches.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("In memory vector store retriever created successfully.")


### Step 5.7: Test the retriever with a sample question

In [ ]:
query = "What is Multi Document Question Answering?"

# Ask the retriever for the chunks that are most relevant to this question.
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n=== Retrieved Chunk {i} ===")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))


### Step 5 workflow summary

This section demonstrated document retrieval using a local, in memory Chroma vector store. It only covers retrieval, not full question answering with a language model, which is introduced in Step 6.

The data flows through the pipeline in the following order.

PDF files, then UnstructuredPDFLoader, then Document objects, then RecursiveCharacterTextSplitter, then text chunks, then the Hugging Face embedding model, then the in memory Chroma vector store, then the retriever, then the relevant document chunks that are returned as the final result.


## Step 6: Loading Multiple PDFs with PyPDFLoader and Storing Them in Astra DB

In Step 5 we created a temporary, in memory vector store purely to demonstrate the basic retrieval workflow. It only exists for the current session and is not meant to be used as the permanent vector database for a real project.

In this step we build a proper, persistent RAG ingestion pipeline using Astra DB as the vector database. Astra DB is a cloud native database built on Apache Cassandra that is designed to store and search vector embeddings efficiently, and the data stored inside it survives even after this notebook is closed.

We load multiple PDF documents using `PyPDFLoader`, split the extracted text into smaller overlapping chunks using `RecursiveCharacterTextSplitter`, generate vector embeddings using the shared Hugging Face embedding model from Step 4, and then store the chunks and their embeddings permanently inside Astra DB.

### Step 6 workflow summary

PDF files, then PyPDFLoader, then Document objects, then RecursiveCharacterTextSplitter, then text chunks, then the Hugging Face embedding model, then the Astra DB vector store, then the retriever, then the relevant document chunks that are returned as the final result.


### Step 6.1: Set the PDF folder path

In [ ]:
import os

# Local folder containing the PDF files used for this section. This can
# be the same folder used in Step 5, or a different one.
pdf_folder_path = r"f:\Multimodal-RAG-Systems\Multi_Document_Rag\data"

# Collect only the file names that end with .pdf.
pdf_files = [
    file_name
    for file_name in os.listdir(pdf_folder_path)
    if file_name.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF file(s):")
print(pdf_files)


### Step 6.2: Create the text splitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter that will break every long PDF document into
# smaller overlapping chunks. A smaller chunk_size than Step 5 is used
# here so that each chunk stored permanently in Astra DB stays compact
# and focused on a single idea.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64
)

print("Text splitter created successfully.")


### Step 6.3: Load and split all PDF files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load every PDF file one at a time and immediately split it into chunks,
# then combine all of the chunks from every file into one single list.
docs_from_pdf = []

for file_name in pdf_files:
    file_path = os.path.join(pdf_folder_path, file_name)

    # Create a loader for the current PDF file.
    loader = PyPDFLoader(file_path)

    # Load the PDF and split it into chunks in a single call.
    chunks = loader.load_and_split(
        text_splitter=splitter
    )

    # Add these chunks to the combined list of all chunks.
    docs_from_pdf.extend(chunks)

print(f"Created {len(docs_from_pdf)} document chunks from {len(pdf_files)} PDF file(s).")


### Step 6.4: Connect to the Astra DB vector store

In [ ]:
from langchain_astradb import AstraDBVectorStore

# Connect to, or automatically create, a collection inside Astra DB.
#
# A single collection stores three things together for every chunk.
# 1. The original text of the chunk.
# 2. The vector embedding generated by the Hugging Face embedding model.
# 3. Metadata about the chunk, such as its source file and page number.
#
# These stored vectors can later be searched using semantic similarity,
# meaning Astra DB can find chunks that mean something similar to a query
# even if they do not share the exact same words.
vstore = AstraDBVectorStore(
    embedding=embedding,
    collection_name="multipdf_vector",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

print("Connected to Astra DB collection: multipdf_vector")


### Step 6.5: Store the PDF chunks in Astra DB

In [ ]:
# Send every PDF chunk to Astra DB.
#
# For each chunk, Astra DB performs two actions behind the scenes.
# 1. The Hugging Face embedding model converts the chunk's text into a vector.
# 2. The text, the vector, and the metadata are saved together in the
#    collection created in the previous cell.
#
# Astra DB returns a unique identifier for every chunk it stores.
inserted_ids_from_pdf = vstore.add_documents(
    docs_from_pdf
)

print(
    f"Inserted {len(inserted_ids_from_pdf)} "
    "document chunks into Astra DB."
)


### Step 6.6: Create the Astra DB retriever

In [ ]:
# Turn the Astra DB vector store into a retriever.
#
# When a query comes in, the retriever performs three steps.
# 1. The query text is converted into an embedding vector.
# 2. Astra DB performs a vector similarity search over every stored chunk.
# 3. The top 3 most relevant chunks are returned.
retriever = vstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Astra DB retriever ready.")


### Step 6.7: Test the retriever against every source document

Three separate questions are asked here on purpose, one aimed at each of the different PDF files that were loaded, so we can confirm the retriever is able to pull relevant chunks out of every single source document rather than just the first one.


In [ ]:
# Ask a sample question aimed at the first PDF.
query = "What is Retrieval Fusions?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n=== Retrieved Chunk {i} ===")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))


In [ ]:
# Ask a sample question aimed at the second PDF.
query = "What is Skopostheorie?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n=== Retrieved Chunk {i} ===")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))


In [ ]:
# Ask a sample question aimed at the third PDF.
query = "What is Multi Document Question Answering?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} relevant chunks.")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n=== Retrieved Chunk {i} ===")
    print(doc.page_content)
    print("\nSource:", doc.metadata.get("source", "Unknown"))


### Step 6.8: Build the prompt template

A prompt template defines the exact instructions that get sent to the language model, together with the retrieved chunks and the user's question.

For every question, the retriever first searches Astra DB and returns the most relevant chunks. Those chunks are inserted wherever the placeholder `{context}` appears in the template, while the user's actual question is inserted wherever `{question}` appears.

The completed prompt is then sent to the language model, which is instructed to answer using only the retrieved context. If the answer is not present anywhere in that context, the model is told to clearly say it could not find the answer rather than inventing one.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt template used for Retrieval Augmented Generation.
prompt_text = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the retrieved context.

If the answer cannot be found in the context, simply say:
"I couldn't find the answer in the provided documents."

Keep your answer clear, concise, and accurate.

Context:
{context}

Question:
{question}

Answer:
"""

rag_prompt = ChatPromptTemplate.from_template(prompt_text)

print("Prompt template created successfully.")


### Step 6.9: Load the Groq API key

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables again in case this cell is run on its own.
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in the .env file.")

print("Groq API key loaded successfully.")


### Step 6.10: Create the Groq language model

In [ ]:
from langchain_groq import ChatGroq

# Initialize the Groq hosted chat model. This is the model that will
# actually write the final answer using the context retrieved from
# Astra DB. temperature is set to 0 so the answers stay focused and
# consistent rather than creative or random.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY,
)

print("Groq LLM ready.")


### Step 6.11: Build the full RAG chain

A chain in LangChain connects several pieces together so that data flows automatically from one step to the next.

This particular chain works as follows. The user's question is sent to the retriever, which returns the context. At the same time the raw question is passed through unchanged using RunnablePassthrough. Both pieces are combined and inserted into the prompt template, the completed prompt is sent to the language model, and finally StrOutputParser converts the model's raw response into a plain text string.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Build the complete Retrieval Augmented Generation chain by piping each
# stage into the next one using the | operator.
rag_chain = (
    {
        "context": retriever,          # fetch relevant chunks for the question
        "question": RunnablePassthrough(),  # pass the raw question through unchanged
    }
    | rag_prompt   # insert both pieces into the prompt template
    | llm          # send the completed prompt to the Groq language model
    | StrOutputParser()  # convert the model's response into plain text
)

print("RAG chain built successfully.")


### Step 6.12: Ask questions using the RAG chain

In [ ]:
query = "What is Advanced ANN Indexing?"

response = rag_chain.invoke(query)

print(response)


In [ ]:
query = "According to Skopostheorie, what is more important than literal translation?"

response = rag_chain.invoke(query)

print(response)


## Step 7: Loading a Folder of Mixed Document Types and Storing Them in Astra DB

In Step 6 the RAG pipeline only worked with PDF documents.

In this step we expand the knowledge base so it can handle an entire folder that contains several different document formats at once, specifically PDF, DOCX, TXT, and PPTX files.

Instead of manually choosing a loader for each individual file type, we use `DirectoryLoader`, which automatically detects the correct loader to use based on each file's extension.

After loading every document, we split them into smaller overlapping chunks using `RecursiveCharacterTextSplitter`, generate embeddings using the same shared Hugging Face embedding model, and store the resulting chunks inside a brand new, separate Astra DB collection dedicated entirely to this mixed document knowledge base.

Using a separate collection keeps this mixed document knowledge base completely independent from the PDF only collection created in Step 6, so the two do not interfere with each other.

### Step 7 workflow summary

Mixed documents, meaning PDF, DOCX, TXT, and PPTX files together, then DirectoryLoader, then Document objects, then RecursiveCharacterTextSplitter, then text chunks, then the Hugging Face embedding model, then the Astra DB vector store, then the retriever, then the relevant document chunks that are returned as the final result.


### Step 7.1: Set the document folder

In [ ]:
import os
import shutil

# Folder containing every mixed document type for this section.
docs_folder_path = r"f:\Multimodal-RAG-Systems\Multi_Document_Rag\data"

# Jupyter sometimes creates a hidden checkpoint folder inside a data
# folder. DirectoryLoader would try to read it as if it were a real
# document, so we remove it first if it exists.
checkpoints_path = os.path.join(
    docs_folder_path,
    ".ipynb_checkpoints"
)

if os.path.isdir(checkpoints_path):
    shutil.rmtree(checkpoints_path)
    print("Removed .ipynb_checkpoints folder.")
else:
    print("No .ipynb_checkpoints folder found.")


### Step 7.2: List every file inside the folder

In [ ]:
import os

# Print every file name found inside the mixed document folder, purely as
# a visual check before loading anything, so we can confirm the folder
# actually contains the PDF, DOCX, TXT, and PPTX files we expect.
for file in os.listdir(docs_folder_path):
    print(file)


### Step 7.3: Load and split the mixed documents

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# DirectoryLoader automatically picks the right loader for every file
# inside the folder based on its file extension, so PDF, DOCX, TXT, and
# PPTX files can all be loaded with this single call.
loader = DirectoryLoader(
    docs_folder_path
)

# Reuse the exact same chunking strategy that was used in Step 6.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64
)

# Load every document in the folder and split it into chunks in one step.
docs = loader.load_and_split(
    text_splitter=splitter
)

print(f"Loaded {len(docs)} document chunks.")


### Step 7.4: Check how many distinct files were actually loaded

In [ ]:
# Look at the metadata attached to every chunk and collect the unique
# set of source file names, so we can confirm that every file type in
# the folder, not just one of them, was successfully loaded.
unique_files = sorted({doc.metadata["source"] for doc in docs})

print(f"Total files loaded: {len(unique_files)}\n")

for i, file in enumerate(unique_files, 1):
    print(f"{i}. {file}")


### Step 7.5: Create a new Astra DB vector store for mixed documents

In [ ]:
from langchain_astradb import AstraDBVectorStore

# Create a brand new, separate Astra DB collection dedicated to the
# mixed document knowledge base, so it stays independent from the
# PDF only collection created earlier in the notebook.
vstore_multidoc = AstraDBVectorStore(
    embedding=embedding,
    collection_name="multidoc_vector",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

print("Connected to Astra DB collection: multidoc_vector")


### Step 7.6: Store the mixed document chunks in Astra DB

In [ ]:
# Generate an embedding for every chunk and store all of them,
# together with their text and metadata, inside the new collection.
inserted_ids = vstore_multidoc.add_documents(
    docs
)

print(
    f"Inserted {len(inserted_ids)} "
    "document chunks into Astra DB."
)


### Step 7.7: Create the retriever for the mixed document collection

In [ ]:
# Create a retriever for this new mixed document collection, again
# returning the top 3 most relevant chunks for any given query.
retriever_multidoc = vstore_multidoc.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")


### Step 7.8: Build the prompt template for the mixed document knowledge base

This prompt template controls how the language model answers questions using context retrieved from the mixed document knowledge base.

For every question, the retriever searches the Astra DB collection and returns the most relevant chunks, regardless of whether those chunks originally came from a PDF, a DOCX file, a TXT file, or a PPTX file. Those chunks fill the `{context}` placeholder, and the user's question fills the `{question}` placeholder.

The model is instructed to answer only using that retrieved context, and to clearly say the answer could not be found if the required information is missing.


In [ ]:
roadmap_prompt_text = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the retrieved context.

If the answer cannot be found in the context, simply say:

"I couldn't find the answer in the provided documents."

Keep your answer clear, concise, and accurate.

Context:
{context}

Question:
{question}

Answer:
"""

print("Mixed document prompt text created successfully.")


### Step 7.9: Build the RAG chain for mixed documents

In [ ]:
# Reuse the same Groq chat model instance created earlier in Step 6.
# Build a second RAG chain that follows the exact same pattern as before,
# but this time using the mixed document retriever and its own prompt.
multidoc_chain = (
    {
        "context": retriever_multidoc,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")


### Step 7.10: Ask questions across different document types

The three questions below are chosen on purpose so that each one is best answered from a different type of file in the mixed document folder, proving that the retriever can search across every format at once.


Example 1, a question best answered using information from a PDF document

In [ ]:
query1 = "What are the stages for using the retriever?"

# Retrieve the chunks most relevant to this question.
retrieved_docs = retriever_multidoc.invoke(query1)

print(f"Retrieved {len(retrieved_docs)} document(s).\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"========== Document {i} ==========")
    print("Source :", doc.metadata.get("source", "Unknown"))
    print("Page   :", doc.metadata.get("page", "N/A"))
    print("\nContent:\n")
    print(doc.page_content)
    print("\n" + "=" * 80 + "\n")


Example 2, a question best answered using information from a PowerPoint, meaning a PPTX document

In [ ]:
query2 = "Give the Roadmap of GenAI?"

retrieved_docs = retriever_multidoc.invoke(query2)

print(f"Retrieved {len(retrieved_docs)} document(s).\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"========== Document {i} ==========")
    print("Source :", doc.metadata.get("source", "Unknown"))
    print("Page   :", doc.metadata.get("page", "N/A"))
    print("\nContent:\n")
    print(doc.page_content)
    print("\n" + "=" * 80 + "\n")


Example 3, a question best answered using information from a plain TXT document

In [ ]:
query3 = "Why does Gared want to turn back?"

retrieved_docs = retriever_multidoc.invoke(query3)

print(f"Retrieved {len(retrieved_docs)} document(s).\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"========== Document {i} ==========")
    print("Source :", doc.metadata.get("source", "Unknown"))
    print("Page   :", doc.metadata.get("page", "N/A"))
    print("\nContent:\n")
    print(doc.page_content)
    print("\n" + "=" * 80 + "\n")


## Conclusion

In this project we built a complete Multi Document Retrieval Augmented Generation system capable of answering questions from several document formats at once, including PDF, DOCX, TXT, and PPTX files.

Instead of limiting the knowledge base to a single document type, `DirectoryLoader` was used to automatically load different file formats straight from a local folder. Every document was split into smaller chunks, converted into vector embeddings using a free Hugging Face embedding model, and stored inside Astra DB, a cloud native vector database.

When a user submits a question, the retriever searches Astra DB for the most relevant chunks. Those chunks are combined with the user's question through a prompt template and sent to a Groq hosted Llama 3.3 70B language model, which generates an accurate answer based only on the retrieved context.

Overall, this project demonstrates how a modern RAG pipeline can build a searchable, multi format knowledge base and answer questions grounded in real documents rather than relying purely on what a language model memorized during training.


## Overall Project Workflow

```text
            Mixed Documents Folder
       (PDF, DOCX, TXT, PPTX Files)
                     │
                     ▼
              DirectoryLoader
     automatically loads every
        supported document type
                     │
                     ▼
              Documents Loaded
                     │
                     ▼
     RecursiveCharacterTextSplitter
       splits documents into chunks
                     │
                     ▼
      Hugging Face Embedding Model
(sentence transformers all MiniLM L6 v2)
      converts each chunk into a vector
                     │
                     ▼
           Astra DB Vector Store
     stores document embeddings securely
                     │
                     ▼
             Astra DB Retriever
   finds the most relevant document chunks
        using semantic similarity search
                     │
                     ▼
             Prompt Template
   combines the retrieved context with the
              user's question
                     │
                     ▼
        Groq Llama 3.3 70B Model
   generates the final grounded answer
```


## Tools and Technologies Used

Programming Language: Python

Development Environment: Visual Studio Code, Jupyter Notebook

Document Loaders: DirectoryLoader, PyPDFLoader, UnstructuredPDFLoader

Supported File Types: PDF, DOCX, TXT, PPTX

Text Splitting: RecursiveCharacterTextSplitter

Embedding Model: Hugging Face, sentence transformers all MiniLM L6 v2

Vector Database: Astra DB by DataStax

Retriever: Astra DB Retriever

Language Model: Groq, Llama 3.3 70B Versatile
